# Myllia - echoes of silenced genes
---

**authors**: [fsb2210](https://www.kaggle.com/fsb2210), [julianc93](https://www.kaggle.com/julianc93)

## Introduction

In this notebook we describe how pre-processed data from several external sources is processed into a final feature matrix.

### External preprocessing

Our start data comes from 4 external datasets, namely, *K562*, *HepG2*, *Jurkat* and *RPE1* (Replogle et al., Nadig et al.).
These raw datasets were applied the same steps as the ones described in the [*Normalization details*](https://www.kaggle.com/competitions/echoes-of-silenced-genes/data) section from the challenge:

* **Input data:** Raw UMI counts for the entire set of genes in the dataset.
* **Normalization:** Log-normalized counts applied uniformly across challenge and external datasets.
  $$
  x_{norm} = \log_2\left(\frac{x_{raw}}{\sum x_{raw}} \times 10,000 + 1\right)
  $$
* **Gene subset:** Data subsetted to the **5,127 target genes** relevant to the challenge.
* **Pseudo-Bulk (PB) aggregation:** Single-cell counts averaged per perturbation condition (arithmetic mean), disregarding batch information.

In [1]:
import os
import time
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from sklearn.decomposition import PCA
from sklearn.impute import KNNImputer
from tqdm import tqdm

Define global options:

In [2]:
# directory with data files
data_dir = "../data"
working_dir = "../data"

# random state integer value for reproducibility concerns
random_state = 42

# neighbors to use for imputation of NaN expressions
n_neighbors = 3

# threshold for considering a gene to have been perturbed
threshold = 0.2

n_latent = 300

## Data loading

We can now load the pseudo-bulk datasets along with an already curated database of protein embeddings for genes

In [3]:
# train & validation sets
raw_train_df = pd.read_csv(f"{data_dir}/training_data_means.csv")
valid_df = pd.read_csv(f"{data_dir}/pert_ids_val.csv")

In [4]:
# K562 dataset
raw_k562_df = pd.read_csv(f"{data_dir}/k562_data_means.csv")
# Hepg2 dataset
raw_hepg2_df = pd.read_csv(f"{data_dir}/hepg2_data_means.csv")
# Jurkat dataset
raw_jurkat_df = pd.read_csv(f"{data_dir}/jurkat_data_means.csv")
# RPE1 dataset
raw_rpe1_df = pd.read_csv(f"{data_dir}/rpe1_data_means.csv")

In [5]:
# load pre-fetched gene embeddings using ESM-2
gene_embeddings_df = pd.read_csv(f"{data_dir}/esm2_gene_embeddings.csv")

## Data preprocessing

Next in line is to manipulate the pseudo-bulk datasets. In particular,

* **Data imputation ($Y_{\rm imp}$)**: Given that some of the target genes are not present in the external dataset, we use a KNN-imputer technique to fill every missing value.
* **Target variable ($Y$):** With the imputed data, we compute differential expression (DE) vectors computed relative to the non-targeting control ($\vec{c}$).
  $$
  \vec{y}_i = \vec{x}_{\rm{perturbed}\_i} - \vec{c}
  $$
  with a final shape of $(N_{\rm samples}, 5127)$.

Grab the name of the output genes:

In [6]:
output_genes = raw_train_df.columns[1:]

Impute missing values in K562 and HepG2 datasets, using nearest beighbors in the training sample:

In [7]:
def impute(y, y_true, n_neighbors):
    # define imputer
    imputer = KNNImputer(n_neighbors=n_neighbors, weights="uniform")
    # stack y (with NaNs) and y_true
    y_stacked = np.vstack([y_true, y])
    y_imputed = imputer.fit_transform(y_stacked)
    y_ = y_imputed.copy()
    return y_[y_true.shape[0]:]

In [8]:
start_time = time.time()

if len(os.listdir("/home/fsbn/.cache/myllia/")) == 5:
    Y_train = np.load("/home/fsbn/.cache/myllia/y_train.npy")
    Y_k562 = np.load("/home/fsbn/.cache/myllia/y_k562.npy")
    Y_hepg2 = np.load("/home/fsbn/.cache/myllia/y_hepg2.npy")
    Y_jurkat = np.load("/home/fsbn/.cache/myllia/y_jurkat.npy")
    Y_rpe1 = np.load("/home/fsbn/.cache/myllia/y_rpe1.npy")
else:
    Y_train = raw_train_df.iloc[:, 1:].values
    Y_k562 = impute(raw_k562_df.iloc[:, 1:].values, Y_train, n_neighbors)
    Y_hepg2 = impute(raw_hepg2_df.iloc[:, 1:].values, Y_train, n_neighbors)
    Y_jurkat = impute(raw_jurkat_df.iloc[:, 1:].values, Y_train, n_neighbors)
    Y_rpe1 = impute(raw_rpe1_df.iloc[:, 1:].values, Y_train, n_neighbors)
    np.save("/home/fsbn/.cache/myllia/y_train.npy", Y_train)
    np.save("/home/fsbn/.cache/myllia/y_k562.npy", Y_k562)
    np.save("/home/fsbn/.cache/myllia/y_hepg2.npy", Y_hepg2)
    np.save("/home/fsbn/.cache/myllia/y_jurkat.npy", Y_jurkat)
    np.save("/home/fsbn/.cache/myllia/y_rpe1.npy", Y_rpe1)

training_time = time.time() - start_time
print(f"Data imputation completed in {training_time:.2f} seconds ({training_time/60:.2f} minutes)")

Data imputation completed in 0.09 seconds (0.00 minutes)


Replace NaNs with newly computed values from imputation

In [9]:
raw_k562_df[output_genes] = Y_k562
raw_hepg2_df[output_genes] = Y_hepg2
raw_jurkat_df[output_genes] = Y_jurkat
raw_rpe1_df[output_genes] = Y_rpe1

Grab the name of the perturbations in each of the datasets

In [10]:
train_pert_genes = raw_train_df.pert_symbol.tolist()[:-1]
k562_pert_genes = raw_k562_df.pert_symbol.tolist()[:-1]
hepg2_pert_genes = raw_hepg2_df.pert_symbol.tolist()[:-1]
jurkat_pert_genes = raw_jurkat_df.pert_symbol.tolist()[:-1]
rpe1_pert_genes = raw_rpe1_df.pert_symbol.tolist()[:-1]

Compute delta expressions with respect to the `non-targeting` row for every dataset:

In [11]:
# train dataset
train_base = raw_train_df.iloc[-1,1:].values
train_df = raw_train_df[output_genes].apply(lambda x: x - train_base, axis=1)
train_df.insert(0, "pert_symbol", raw_train_df["pert_symbol"].copy())
train_df.drop([train_df.shape[0]-1], inplace=True)
train_df["source"] = "myllia"

# K562 dataset
k562_base = raw_k562_df.iloc[-1,1:].values
k562_df = raw_k562_df[output_genes].apply(lambda x: x - k562_base, axis=1)
k562_df.insert(0, "pert_symbol", raw_k562_df["pert_symbol"].copy())
k562_df.drop([k562_df.shape[0]-1], inplace=True)
k562_df["source"] = "k562"

# HepG2 dataset
hepg2_base = raw_hepg2_df.iloc[-1,1:].values
hepg2_df = raw_hepg2_df[output_genes].apply(lambda x: x - hepg2_base, axis=1)
hepg2_df.insert(0, "pert_symbol", raw_hepg2_df["pert_symbol"].copy())
hepg2_df.drop([hepg2_df.shape[0]-1], inplace=True)
hepg2_df["source"] = "hepg2"

# Jurkat dataset
jurkat_base = raw_jurkat_df.iloc[-1,1:].values
jurkat_df = raw_jurkat_df[output_genes].apply(lambda x: x - jurkat_base, axis=1)
jurkat_df.insert(0, "pert_symbol", raw_jurkat_df["pert_symbol"].copy())
jurkat_df.drop([jurkat_df.shape[0]-1], inplace=True)
jurkat_df["source"] = "jurkat"

# RPE1 dataset
rpe1_base = raw_rpe1_df.iloc[-1,1:].values
rpe1_df = raw_rpe1_df[output_genes].apply(lambda x: x - rpe1_base, axis=1)
rpe1_df.insert(0, "pert_symbol", raw_rpe1_df["pert_symbol"].copy())
rpe1_df.drop([rpe1_df.shape[0]-1], inplace=True)
rpe1_df["source"] = "rpe1"

train_df.shape, k562_df.shape, hepg2_df.shape, jurkat_df.shape, rpe1_df.shape

((80, 5129), (2057, 5129), (2393, 5129), (2393, 5129), (2393, 5129))

## Feature and target engineering

We move on to creating the signatures for every single gene in the external datasets.

### Gene signatures

* **Signature computation:** PB-DE vectors computed for each perturbation in all the external datasets using the same normalization.
* **Gene-Level Aggregation:** PB-DE vectors grouped by gene symbol across all external cell lines and averaged a perturbation signature.
  $$
  \vec{s}_{\rm gene} = \frac{1}{K} \sum_{k=1}^{K} \vec{y}_{\rm{external}\_k}
  $$
* **Dimensionality reduction:** Principal Component Analysis (PCA) fitted **only*8 on the aggregated external signature matrix to prevent data leakage during training.
  * **Components:** $n_{\rm components} = $ `n_latent`.
* **Lookup table:** Created the mapping `Gene Symbol` $\rightarrow$ `PCA Vector (300,)`.

### Protein embeddings

We used an evolutionary scale modeling from *Meta-AI* to convert aminoacid sequences into embedding vectors.

* **Model:** `esm2_t33_650M_UR50D`.
* **Input:** Amino acid sequences for each perturbed gene.
* **Output:** Fixed-length protein embedding vector of shape $(1, 1280)$.

First, let's concatenate external datasets into a single one:

In [12]:
external_df = pd.concat([k562_df, hepg2_df, jurkat_df, rpe1_df], axis=0)

Next, we group by the perturbation symbols (`pert_symbol`):

In [13]:
gene_signatures = external_df.groupby("pert_symbol")[output_genes].mean()
gene_signatures.shape

(2395, 5127)

We can now reduce the dimensions using PCA:

In [14]:
pca = PCA(n_components=n_latent)
signature_matrix = gene_signatures.values
pca_features = pca.fit_transform(signature_matrix)

# create dataframe with PCA transformed values
ext_ds_feats = pd.DataFrame(pca_features, columns=[f"pca_{k}" for k in range(pca_features.shape[1])])
ext_ds_feats.insert(0, "pert_symbol", gene_signatures.index.tolist())
pca_lookup = ext_ds_feats.set_index("pert_symbol")

In [15]:
overlap = len(set(train_df["pert_symbol"]) & set(ext_ds_feats["pert_symbol"]))
print(f"Overlap: {overlap} / 80 genes")

Overlap: 31 / 80 genes


### Final feature matrix

We are now ready to build the final feature matrix for the training and validation datasets.

* **Hybrid Vector:** For each perturbation (training or validation), features concatenated as follows:
  $$
  \vec{f}_{\rm final} = [\vec{f}_{\rm ESM2} \oplus \vec{f}_{\rm PCA} \oplus f_{\rm mask}]
  $$
  * $\vec{f}_{\rm ESM2}$: Protein embedding (1280 dims).
  * $\vec{f}_{\rm PCA}$: External PCA signature (`n_latent` dims). If gene missing from external lookup, imputed with zeros.
  * $f_{\rm mask}$: Binary indicator (1 if PCA signature available, 0 otherwise).
* **Total Feature Dimension:** for `n_latent = 300`, $1280 + 300 + 1 = \mathbf{1581}$.

In [16]:
final_features = []
gene_list = train_df["pert_symbol"].tolist()
for gene in gene_list:
    # 1. get protein embeddings
    prot_emb = gene_embeddings_df[gene_embeddings_df["pert_symbol"] == gene]
    if len(prot_emb) == 0:
        print(f"could ont find embeddings for gene `{gene}`")
        # fallback if embedding missing
        emb_vec = np.zeros(1280)
    else:
        emb_vec = prot_emb[gene_embeddings_df.columns[1:]].to_numpy()[0]  # prot_emb.values[0][1:]
    
    # 2. get PCA signature
    if gene in pca_lookup.index:
        pca_vec = pca_lookup.loc[gene].values
        has_pca = 1.0
    else:
        # fallback: use zeros
        pca_vec = np.zeros(300)  # pca_lookup.mean().values 
        has_pca = 0.0
    
    # 3. Concatenate
    full_vector = np.concatenate([emb_vec, pca_vec, [has_pca]])
    final_features.append(full_vector)

In [17]:
X = np.array(final_features)
Y = train_df[output_genes].values

Same as before but for the validation set

In [18]:
final_features = []
valid_df = valid_df.rename({"pert": "pert_symbol"}, axis=1)
val_gene_list = valid_df["pert_symbol"].tolist()
for gene in val_gene_list:
    # 1. get protein embeddings
    prot_emb = gene_embeddings_df[gene_embeddings_df["pert_symbol"] == gene]
    if len(prot_emb) == 0:
        print(f"could ont find embeddings for gene `{gene}`")
        # fallback if embedding missing
        emb_vec = np.zeros(1280)
    else:
        emb_vec = prot_emb[gene_embeddings_df.columns[1:]].to_numpy()[0]  # prot_emb.values[0][1:]
    
    # 2. get PCA signature
    if gene in pca_lookup.index:
        pca_vec = pca_lookup.loc[gene].values
        has_pca = 1.0
    else:
        # fallback: use zeros
        pca_vec = np.zeros(300)  # pca_lookup.mean().values 
        has_pca = 0.0
    
    # 3. Concatenate
    full_vector = np.concatenate([emb_vec, pca_vec, [has_pca]], dtype=float)
    final_features.append(full_vector)

In [19]:
X_val = np.array(final_features)

In [20]:
print(f"- training feature shape: {X.shape}, training output shape: {Y.shape}")
print(f"- validation feature shape: {X_val.shape}")

- training feature shape: (80, 1581), training output shape: (80, 5127)
- validation feature shape: (60, 1581)


### Final features storage

Finally, we save features to be used later on

In [21]:
np.save("../data/processed/X_esm2_pca_train.npy", X)
np.save("../data/processed/X_esm2_pca_val.npy", X_val)
np.save("../data/processed/Y_esm2_pca_train.npy", Y)

---

## Bonus: overlap analysis between training and validation sets

In [22]:
train_overlap = len(set(train_df["pert_symbol"]) & set(ext_ds_feats["pert_symbol"]))
valid_overlap = len(set(valid_df["pert_symbol"]) & set(ext_ds_feats["pert_symbol"]))

print(f"- training genes overlap: {train_overlap} / {train_df.shape[0]} ({train_overlap/train_df.shape[0]*100:.1f}%)")
print(f"- validation genes overlap: {valid_overlap} / {train_df.shape[0]} ({valid_overlap/valid_df.shape[0]*100:.1f}%)")

- training genes overlap: 31 / 80 (38.8%)
- validation genes overlap: 22 / 80 (36.7%)


- The proportion of genes with external PCA features is nearly identical (~37-39%) between training and validation. Thus cross-validation results on the 80 training samples should generalize reliably to the 60 validation genes.

- There is no distribution shift so we shouldn't have to worry about the model learning to rely on PCA features during training, only to find they're missing at test time (or vice versa).

- Around 37% of the predictions of the validation set should benefit from external prior.